# Sampling

`gemini-3.5-flash-lite` through `generate_content`. Logprobs are requested the way the API documents them. The later cells are what to do when that request is refused, and the check that temperature actually changed the distribution.


## Cell 1 — client, with top-5 logprobs turned on

In [3]:
import math
import sys
from collections import Counter
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "cli").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from google.genai.errors import ClientError

from cli import config
from cli.llm import complete, response_text

OPEN = "Reply with one random integer from 0 to 99999. Digits only."
SURE = "What is 6*7? Reply with digits only."


def ask(prompt, **kwargs):
    # response_logprobs asks for the chosen token's logprob.
    # logprobs=5 asks for the top 5 alternatives at each position.
    max_tokens = kwargs.pop("max_tokens", 8)
    return complete(
        prompt,
        response_logprobs=True,
        logprobs=5,
        max_tokens=max_tokens,
        **kwargs,
    )


def position_rows(response):
    result = response.candidates[0].logprobs_result
    if result is None or not result.top_candidates:
        return []
    rows = []
    for top in result.top_candidates:
        rows.append([
            (candidate.token, math.exp(candidate.log_probability))
            for candidate in (top.candidates or [])
        ])
    return rows


def show_positions(response):
    rows = position_rows(response)
    if not rows:
        print("no per-token distribution on the response")
        print("text:", repr(response_text(response)))
        return rows
    for index, row in enumerate(rows):
        print(f"position {index}")
        for token, probability in row:
            print(f"  {probability:8.4f}  {token!r}")
    return rows


print("model:", config.MODEL)


model: gemini-3.5-flash-lite


## Cell 2 — one prompt, top 5 at each output position

`log(p)` comes back. `exp` turns it into a probability. These are only the top 5, so they will not sum to 1.


In [5]:
logprob_error = None
try:
    first = ask(OPEN, temperature=1.0)
    show_positions(first)
except ClientError as exc:
    first = None
    logprob_error = exc
    print(f"{exc.code} {exc.status}")
    print(exc.message)


400 INVALID_ARGUMENT
Logprobs is not enabled for this model


## Cell 3 — the same prompt at four temperatures

If logprobs came back, the four position-0 distributions are printed together. They are the right comparison: later positions are conditioned on different chosen tokens, so they are not the same list reshaped.

If logprobs were refused, each temperature is sampled a few times instead. A working temperature-0 is one repeated string. A working temperature-2 spreads out.


In [7]:
from cli.sampling import apply_temperature

TEMPERATURES = (0, 0.7, 1.0, 2.0)
api_rows = {}
samples = {}

for temperature in TEMPERATURES:
    print(f"\n=== temperature {temperature}")
    try:
        response = ask(OPEN, temperature=temperature)
        api_rows[temperature] = show_positions(response)
    except ClientError as exc:
        api_rows[temperature] = None
        print(f"logprobs refused: {exc.code} {exc.message.splitlines()[0]}")
    drawn = [
        response_text(complete(OPEN, max_tokens=16, temperature=temperature)).strip()
        for _ in range(4)
    ]
    samples[temperature] = drawn
    print("samples:", drawn)



=== temperature 0
logprobs refused: 400 Logprobs is not enabled for this model
samples: ['58231', '48291', '48291', '73429']

=== temperature 0.7
logprobs refused: 400 Logprobs is not enabled for this model
samples: ['74321', '48291', '47892', '74821']

=== temperature 1.0
logprobs refused: 400 Logprobs is not enabled for this model
samples: ['47823', '48291', '48291', '48291']

=== temperature 2.0
logprobs refused: 400 Logprobs is not enabled for this model
samples: ['48291', '48291', '48291', '48291']


## Cell 4 — apply the temperature formula yourself

Temperature divides logits before softmax. A logit is `log(p) + C`, and the constant cancels, so a temperature-1 distribution rescales as `p ** (1/T)`, then renormalise. Temperature 0 keeps only the winner.

The base here is the temperature-0 distribution from the API when we have one. Otherwise the notes' 70/20/10 example is the base, and the thing compared to the API is the seeded draw in the seed cell: same seed, different temperatures, which must disagree if the masses moved.


In [9]:
def as_dict(rows):
    return {token: probability for token, probability in rows[0]} if rows else None


base = as_dict(api_rows.get(0))
source = "API temperature 0, position 0"
if base is None or sum(1 for p in base.values() if p > 0) <= 1:
    # A collapsed temperature-0 list has no logits left to divide.
    base = as_dict(api_rows.get(1.0)) or {"Paris": 0.70, "located": 0.20, "the": 0.10}
    source = "API temperature 1, position 0" if api_rows.get(1.0) else "notes example, 70/20/10"

print("base:", source)
for token, probability in base.items():
    print(f"  {probability:8.4f}  {token!r}")

print("\nformula")
predicted = {}
for temperature in TEMPERATURES:
    predicted[temperature] = apply_temperature(base, temperature)
    print(f"  T={temperature}")
    for token, probability in predicted[temperature].items():
        print(f"    {probability:8.4f}  {token!r}")

print("\nagainst the API")
compared = False
for temperature in TEMPERATURES:
    actual = as_dict(api_rows.get(temperature))
    if not actual:
        continue
    compared = True
    print(f"  T={temperature}")
    keys = list(dict.fromkeys([*predicted[temperature], *actual]))
    for token in keys:
        expect = predicted[temperature].get(token)
        got = actual.get(token)
        expect_s = f"{expect:8.4f}" if expect is not None else "       -"
        got_s = f"{got:8.4f}" if got is not None else "       -"
        print(f"    {token!r:20} formula {expect_s}   api {got_s}")

if not compared:
    print("  no API distribution came back, so there is nothing to subtract.")
    print("  sample counts, which is the picker, not the list:")
    for temperature, drawn in samples.items():
        print(f"    T={temperature}: {dict(Counter(drawn))}")
    print("  the seed cell is the controlled comparison.")


base: notes example, 70/20/10
    0.7000  'Paris'
    0.2000  'located'
    0.1000  'the'

formula
  T=0
      1.0000  'Paris'
      0.0000  'located'
      0.0000  'the'
  T=0.7
      0.8136  'Paris'
      0.1359  'located'
      0.0505  'the'
  T=1.0
      0.7000  'Paris'
      0.2000  'located'
      0.1000  'the'
  T=2.0
      0.5229  'Paris'
      0.2795  'located'
      0.1976  'the'

against the API
  no API distribution came back, so there is nothing to subtract.
  sample counts, which is the picker, not the list:
    T=0: {'58231': 1, '48291': 2, '73429': 1}
    T=0.7: {'74321': 1, '48291': 1, '47892': 1, '74821': 1}
    T=1.0: {'47823': 1, '48291': 3}
    T=2.0: {'48291': 4}
  the seed cell is the controlled comparison.


## Cell 5 — a token the model is sure about, at temperature 2

The formula: a 99% token at temperature 2 is still about 91%. It usually survives. Temperature flattens a close race. It does not dethrone a token that was almost certain.


In [11]:
print(
    "99% at temperature 2 ->",
    round(apply_temperature({"42": 0.99, "other": 0.01}, 2.0)["42"], 4),
)
for temperature in (0, 2.0):
    drawn = [
        response_text(complete(SURE, max_tokens=8, temperature=temperature)).strip()
        for _ in range(3)
    ]
    print(f"T={temperature}: {drawn}")


99% at temperature 2 -> 0.9087
T=0: ['42', '42', '42']
T=2.0: ['42', '42', '42']


## Cell 6 — top_p at 1.0 and at 0.5, temperature fixed

Cutoff and flattening are different operations.

Flattening (temperature) keeps every token and moves the percentages. Cutoff (top_p) sorts, accumulates, and drops whatever is past `p`. On a 70/20/10 list, top_p 0.5 stops at the first token, because 0.70 is already past 0.5. Temperature 2 keeps all three.

The live check holds the seed still, so a real cutoff changes which token is drawn and a no-op cutoff does not.


In [13]:
flat = apply_temperature({"Paris": 0.70, "located": 0.20, "the": 0.10}, 2.0)
print("temperature 2 keeps all three:", {k: round(v, 3) for k, v in flat.items()})
print("top_p 0.5 on that 70/20/10 list keeps only Paris (0.70 >= 0.5)")

SEED = 7
for top_p in (1.0, 0.5):
    text = response_text(
        complete(OPEN, max_tokens=16, temperature=1.0, top_p=top_p, seed=SEED)
    ).strip()
    print(f"top_p={top_p} seed={SEED}: {text}")


temperature 2 keeps all three: {'Paris': 0.523, 'located': 0.279, 'the': 0.198}
top_p 0.5 on that 70/20/10 list keeps only Paris (0.70 >= 0.5)
top_p=1.0 seed=7: 48216
top_p=0.5 seed=7: 48216


## Seed

Same prompt, same seed, twice. Then a different seed. Then the original seed at temperature 2.

A seed fixes the draw from whatever distribution the picker is using. It does not fix temperature. If temperature is actually applied, the same seed at temperature 2 falls in a different bin, because the bin widths changed, and the text changes. If the text does not change, temperature was not applied.


In [15]:
def seeded(seed, temperature):
    return response_text(
        complete(OPEN, max_tokens=16, seed=seed, temperature=temperature)
    ).strip()


same_a = seeded(7, 1.0)
same_b = seeded(7, 1.0)
other = seeded(99, 1.0)
hot_a = seeded(7, 2.0)
hot_b = seeded(7, 2.0)
cold = seeded(7, 0)

print("seed 7, temperature 1, twice: ", same_a, same_b)
print("seed 99, temperature 1:        ", other)
print("seed 7, temperature 2, twice:  ", hot_a, hot_b)
print("seed 7, temperature 0:         ", cold)
print("same seed stable:              ", same_a == same_b == hot_a == hot_b)
print("different seed differs:        ", other != same_a)
print("temperature changed the draw:  ", len({same_a, hot_a, cold}) > 1)


seed 7, temperature 1, twice:  48216 48216
seed 99, temperature 1:         48291
seed 7, temperature 2, twice:   48216 48216
seed 7, temperature 0:          48216
same seed stable:               True
different seed differs:         True
temperature changed the draw:   False
